# Retry Failed Markets Pipeline

Loads `half_done_openai_output.csv`, filters out:
- Markets already closed (close_time before Feb 15 midnight UTC)
- Successful predictions (keeps only failed/missing market-arm combos)

Then re-runs only those through the same 3-arm pipeline with rate-limit pacing.

In [3]:
pip install openai

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 1.1 MB 3.9 MB/s eta 0:00:01
     |████████████████████████████████| 309 kB 50.2 MB/s eta 0:00:01
     |████████████████████████████████| 463 kB 30.7 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 22.8 MB/s eta 0:00:01
     |████████████████████████████████| 1.9 MB 35.7 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import re
import asyncio
import random
from datetime import datetime, timezone
import pandas as pd
from openai import AsyncOpenAI

# Config
DATA_PATH = "data/markets_microstructure_v2_v3_merged.csv"
HALF_DONE_PATH = "data/output/half_done_openai_output.csv"
OUT_DIR = "data/output"
OUT_PATH = os.path.join(OUT_DIR, "predictions_retry_completed.csv")
MODEL = "gpt-4o-mini"
CONCURRENCY = 4  # Lower concurrency to avoid rate limits
TEMPERATURE = 0
DELAY_BETWEEN_CALLS = 0.15  # 150ms delay to stay under RPM limits

os.makedirs(OUT_DIR, exist_ok=True)

# Load API key
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        api_key = os.environ.get("OPENAI_API_KEY")
    except:
        pass
if not api_key:
    api_key = input("Enter OPENAI_API_KEY: ").strip()

client = AsyncOpenAI(api_key=api_key)

print(f"Model: {MODEL}")
print(f"Concurrency: {CONCURRENCY}, Delay: {DELAY_BETWEEN_CALLS}s")

Model: gpt-4o-mini
Concurrency: 4, Delay: 0.15s


In [5]:
# Load data
df_all = pd.read_csv(DATA_PATH)
df_done = pd.read_csv(HALF_DONE_PATH)

print(f"Total markets in dataset: {len(df_all)}")
print(f"Rows in half-done output: {len(df_done)}")
print(f"Successful predictions: {df_done['error'].isna().sum()}")
print(f"Failed predictions: {df_done['error'].notna().sum()}")

Total markets in dataset: 1844
Rows in half-done output: 2800
Successful predictions: 2608
Failed predictions: 192


In [6]:
# Step 1: Filter out markets that have already closed
CUTOFF = "2026-02-15T00:00:00+00:00"

df_open = df_all[df_all['close_time'] > CUTOFF].copy()
print(f"Markets still open after {CUTOFF}: {len(df_open)}")
print(f"Markets already closed (filtered out): {len(df_all) - len(df_open)}")

Markets still open after 2026-02-15T00:00:00+00:00: 1676
Markets already closed (filtered out): 168


In [7]:
# Step 2: Find which (market_ticker, arm) combos already succeeded
successful = df_done[df_done['error'].isna()]
done_pairs = set(zip(successful['market_ticker'], successful['arm']))
print(f"Successfully completed (market, arm) pairs: {len(done_pairs)}")

# Step 3: Build list of (market_ticker, arm) combos that need retrying
all_arms = ['baseline', 'volume', 'full_technical']
retry_tasks = []

for _, row in df_open.iterrows():
    for arm in all_arms:
        if (row['market_ticker'], arm) not in done_pairs:
            retry_tasks.append((row['market_ticker'], arm))

retry_tickers = set(t[0] for t in retry_tasks)

print(f"\nMarket-arm combos to retry: {len(retry_tasks)}")
print(f"Unique markets to retry: {len(retry_tickers)}")
print(f"\nBreakdown by arm:")
for arm in all_arms:
    count = sum(1 for t in retry_tasks if t[1] == arm)
    print(f"  {arm}: {count}")

Successfully completed (market, arm) pairs: 2608

Market-arm combos to retry: 2502
Unique markets to retry: 904

Breakdown by arm:
  baseline: 829
  volume: 837
  full_technical: 836


In [ ]:
# API functions with retry logic and rate pacing

def parse_probability(text):
    if not text or not text.strip():
        raise ValueError("Empty response")
    match = re.search(r'([01]?\.\d+|[01])', text.strip())
    if not match:
        raise ValueError(f"No probability in: '{text}'")
    p = float(match.group(1))
    if p > 1:
        p = p / 100
    if not (0 <= p <= 1):
        raise ValueError(f"Out of range: {p}")
    return p

async def query_market(row, arm_name, ticker, semaphore, max_retries=3):
    async with semaphore:
        # Rate pacing
        await asyncio.sleep(DELAY_BETWEEN_CALLS)
        
        prompt = ARMS[arm_name](row)
        raw = None
        p_yes = None
        error = None
        
        for attempt in range(1, max_retries + 1):
            try:
                response = await client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=TEMPERATURE,
                )
                raw = response.choices[0].message.content.strip()
                p_yes = parse_probability(raw)
                error = None
                break
            except ValueError as e:
                error = f"Parse: {str(e)}"
                break
            except Exception as e:
                error = f"API: {str(e)}"
                if attempt < max_retries:
                    await asyncio.sleep((2 ** attempt) + random.random())
                else:
                    break
        
        return {
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'model': MODEL,
            'arm': arm_name,
            'event_ticker': row['event_ticker'],
            'market_ticker': ticker,
            'title': row['event_title'],
            'mid_yes': row['mid_yes'],
            'raw_response': raw,
            'p_yes': p_yes,
            'error': error,
            'has_technical': pd.notna(row.get('return_24h')),
            'attempts': attempt
        }

print(f"API functions loaded (delay: {DELAY_BETWEEN_CALLS}s per call, max 3 retries)")

In [ ]:
# Test with 3 retry tasks first
async def test_retry():
    sem = asyncio.Semaphore(CONCURRENCY)
    test_tasks = retry_tasks[:3]
    results = []
    
    # Build a dict for fast lookup: ticker -> row dict
    market_lookup = {row['market_ticker']: row for _, row in df_open.iterrows()}
    
    print(f"Testing {len(test_tasks)} retry tasks...\n")
    
    for ticker, arm in test_tasks:
        row = market_lookup[ticker]
        result = await query_market(row, arm, ticker, sem)
        results.append(result)
        status = "OK" if not result['error'] else f"ERROR: {result['error']}"
        p_str = f"{result['p_yes']:.3f}" if result['p_yes'] else "None"
        print(f"  {ticker[:40]:40s} | {arm:15s} | {p_str} | {status}")
    
    success = sum(1 for r in results if not r['error'])
    print(f"\nTest: {success}/{len(results)} successful")
    return results

test_results = await test_retry()

In [ ]:
# Full retry pipeline with progress tracking and incremental saves
async def run_retry_pipeline():
    sem = asyncio.Semaphore(CONCURRENCY)
    
    # Build a dict for fast lookup: ticker -> row dict
    market_lookup = {row['market_ticker']: row for _, row in df_open.iterrows()}
    
    total_calls = len(retry_tasks)
    print(f"Starting retry pipeline: {total_calls} calls")
    print(f"Started: {datetime.now().strftime('%H:%M:%S')}")
    print(f"Estimated time: ~{total_calls * DELAY_BETWEEN_CALLS / CONCURRENCY / 60:.1f} min\n")
    
    # Create all async tasks
    async_tasks = []
    for ticker, arm in retry_tasks:
        row = market_lookup[ticker]
        async_tasks.append(query_market(row, arm, ticker, sem))
    
    # Process with progress tracking
    results = []
    successful = 0
    start_time = datetime.now()
    
    for i, task in enumerate(asyncio.as_completed(async_tasks), 1):
        result = await task
        results.append(result)
        
        if result['error'] is None:
            successful += 1
        
        # Progress every 100 calls
        if i % 100 == 0 or i == total_calls:
            elapsed = (datetime.now() - start_time).total_seconds()
            rate = i / max(elapsed, 1) * 60
            remaining = (total_calls - i) / max(rate, 1)
            print(f"[{i:4d}/{total_calls}] {successful}/{i} OK ({successful/i*100:.1f}%) | "
                  f"{rate:.0f} calls/min | ETA: {remaining:.1f} min")
        
        # Incremental save every 500
        if i % 500 == 0:
            pd.DataFrame(results).to_csv(OUT_PATH + '.tmp', index=False)
            print(f"  -> Checkpoint saved ({i} results)")
    
    print(f"\nDone: {datetime.now().strftime('%H:%M:%S')}")
    print(f"Final: {successful}/{total_calls} successful ({successful/total_calls*100:.1f}%)")
    
    return pd.DataFrame(results)

df_retry = await run_retry_pipeline()
df_retry.to_csv(OUT_PATH, index=False)
print(f"\nSaved {len(df_retry)} retry predictions to {OUT_PATH}")

In [12]:
# Test with 3 retry tasks first
async def test_retry():
    sem = asyncio.Semaphore(CONCURRENCY)
    test_tasks = retry_tasks[:3]
    results = []
    
    # Index markets by ticker for fast lookup
    market_lookup = df_open.set_index('market_ticker')
    
    print(f"Testing {len(test_tasks)} retry tasks...\n")
    
    for ticker, arm in test_tasks:
        row = market_lookup.loc[ticker]
        result = await query_market(row, arm, sem)
        results.append(result)
        status = "OK" if not result['error'] else f"ERROR: {result['error']}"
        p_str = f"{result['p_yes']:.3f}" if result['p_yes'] else "None"
        print(f"  {ticker[:40]:40s} | {arm:15s} | {p_str} | {status}")
    
    success = sum(1 for r in results if not r['error'])
    print(f"\nTest: {success}/{len(results)} successful")
    return results

test_results = await test_retry()

Testing 3 retry tasks...



KeyError: 'market_ticker'

In [ ]:
# Full retry pipeline with progress tracking and incremental saves
async def run_retry_pipeline():
    sem = asyncio.Semaphore(CONCURRENCY)
    market_lookup = df_open.set_index('market_ticker')
    
    total_calls = len(retry_tasks)
    print(f"Starting retry pipeline: {total_calls} calls")
    print(f"Started: {datetime.now().strftime('%H:%M:%S')}")
    print(f"Estimated time: ~{total_calls * DELAY_BETWEEN_CALLS / CONCURRENCY / 60:.1f} min\n")
    
    # Create all async tasks
    async_tasks = []
    for ticker, arm in retry_tasks:
        row = market_lookup.loc[ticker]
        async_tasks.append(query_market(row, arm, sem))
    
    # Process with progress tracking
    results = []
    successful = 0
    start_time = datetime.now()
    
    for i, task in enumerate(asyncio.as_completed(async_tasks), 1):
        result = await task
        results.append(result)
        
        if result['error'] is None:
            successful += 1
        
        # Progress every 100 calls
        if i % 100 == 0 or i == total_calls:
            elapsed = (datetime.now() - start_time).total_seconds()
            rate = i / max(elapsed, 1) * 60
            remaining = (total_calls - i) / max(rate, 1)
            print(f"[{i:4d}/{total_calls}] {successful}/{i} OK ({successful/i*100:.1f}%) | "
                  f"{rate:.0f} calls/min | ETA: {remaining:.1f} min")
        
        # Incremental save every 500
        if i % 500 == 0:
            pd.DataFrame(results).to_csv(OUT_PATH + '.tmp', index=False)
            print(f"  -> Checkpoint saved ({i} results)")
    
    print(f"\nDone: {datetime.now().strftime('%H:%M:%S')}")
    print(f"Final: {successful}/{total_calls} successful ({successful/total_calls*100:.1f}%)")
    
    return pd.DataFrame(results)

df_retry = await run_retry_pipeline()
df_retry.to_csv(OUT_PATH, index=False)
print(f"\nSaved {len(df_retry)} retry predictions to {OUT_PATH}")

In [11]:
# Merge: combine original successes + retry results
df_original_success = df_done[df_done['error'].isna()].copy()

# Only keep retry successes
df_retry_success = df_retry[df_retry['error'].isna()].copy()

# Align columns
shared_cols = ['timestamp', 'model', 'arm', 'event_ticker', 'market_ticker', 
               'title', 'mid_yes', 'raw_response', 'p_yes', 'error', 'has_technical']

# Add mid_yes to original if missing
if 'mid_yes' not in df_original_success.columns:
    mid_map = df_all.set_index('market_ticker')['mid_yes'].to_dict()
    df_original_success['mid_yes'] = df_original_success['market_ticker'].map(mid_map)

# Use only columns that exist in both
cols_to_use = [c for c in shared_cols if c in df_original_success.columns and c in df_retry_success.columns]

df_combined = pd.concat([df_original_success[cols_to_use], df_retry_success[cols_to_use]], ignore_index=True)

# Filter combined to only open markets
open_tickers = set(df_open['market_ticker'])
df_combined = df_combined[df_combined['market_ticker'].isin(open_tickers)]

COMBINED_PATH = os.path.join(OUT_DIR, "predictions_combined_final.csv")
df_combined.to_csv(COMBINED_PATH, index=False)

print(f"Combined results:")
print(f"  From original run: {len(df_original_success[df_original_success['market_ticker'].isin(open_tickers)])}")
print(f"  From retry run: {len(df_retry_success)}")
print(f"  Total successful: {len(df_combined)}")
print(f"  Unique markets: {df_combined['market_ticker'].nunique()}")
print(f"\nSaved to {COMBINED_PATH}")

NameError: name 'df_retry' is not defined

In [ ]:
# Summary stats
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

print(f"\nTotal predictions: {len(df_combined)}")
print(f"Unique markets: {df_combined['market_ticker'].nunique()}")

print(f"\nBy arm:")
for arm in ['baseline', 'volume', 'full_technical']:
    arm_df = df_combined[df_combined['arm'] == arm]
    print(f"  {arm:17s}: {len(arm_df)} predictions, "
          f"{arm_df['market_ticker'].nunique()} unique markets")

# Markets with all 3 arms complete
arm_counts = df_combined.groupby('market_ticker')['arm'].nunique()
complete = (arm_counts == 3).sum()
partial = (arm_counts < 3).sum()
print(f"\nMarkets with all 3 arms: {complete}")
print(f"Markets with partial arms: {partial}")

# Retry failures
retry_failures = df_retry[df_retry['error'].notna()]
if len(retry_failures) > 0:
    print(f"\nRetry failures: {len(retry_failures)}")
    print(retry_failures['error'].value_counts().head(5))